<a href="https://colab.research.google.com/github/eduardozurek/ia/blob/main/_007_aprendizaje_supervisado/_007_arboles_de_decision/_007_01_ccp_alpha_DecisionTreeClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# `ccp_alpha` en `DecisionTreeClassifier`

**Pregunta:** Explícame el parámetro **`ccp_alpha`** de `DecisionTreeClassifier` de sklearn.

**Respuesta:** `ccp_alpha` controla la poda del árbol después de haberlo construido, mediante *Minimal Cost-Complexity Pruning* (poda por costo-complejidad).

La idea central es equilibrar dos objetivos:

$$\boxed{\text{ajustar bien los datos}\quad\longleftrightarrow\quad\text{mantener un árbol simple}}$$

## ¿Qué hace `ccp_alpha`?

Scikit-learn considera una función de costo aproximadamente de la forma:

$$R_\alpha(T)=R(T)+\alpha |T_L|$$

donde:

- $T$: árbol considerado.
- $R(T)$: impureza total de las hojas del árbol.
- $|T_L|$: número de hojas.
- $\alpha$: parámetro de penalización, es decir, `ccp_alpha`.

Por tanto, `ccp_alpha` penaliza la complejidad del árbol. Cuando `ccp_alpha = 0`, no hay penalización adicional por complejidad. El árbol puede crecer hasta donde permitan otros parámetros como `max_depth` y `min_samples_split`.

$$\boxed{\uparrow\texttt{ccp\_alpha}\quad\Rightarrow\quad\text{más poda}\quad\Rightarrow\quad\text{árbol más pequeño}}$$

In [ ]:
from sklearn.tree import DecisionTreeClassifier

# Ejemplo de configuración; requiere X_train e y_train definidos previamente.
modelo = DecisionTreeClassifier(ccp_alpha=0.01, random_state=42)
# modelo.fit(X_train, y_train)

## ¿Por qué se necesita?

Un árbol muy profundo puede tener un excelente desempeño sobre los datos de entrenamiento, pero estar **sobreajustado**. Con `ccp_alpha > 0`, algunos subárboles cuya contribución a reducir la impureza no justifica su complejidad son eliminados. Al aumentar más `ccp_alpha`, el árbol se simplifica y eventualmente puede quedar solo el nodo raíz.

## ¿Cómo decide qué ramas eliminar?

Para cada nodo interno, se compara la impureza si ese nodo se convierte en hoja con la impureza de las hojas actuales de su subárbol. Se puede definir un **alpha efectivo**:

$$\boxed{\alpha_{\mathrm{eff}}(t)=\frac{R(t)-R(T_t)}{|T_t|-1}}$$

donde $t$ es un nodo interno, $T_t$ es el subárbol que comienza en $t$, $R(t)$ es la impureza si $t$ se convierte en hoja, $R(T_t)$ es la impureza de las hojas actuales del subárbol y $|T_t|$ es su número de hojas.

Los subárboles con menor $\alpha_{\mathrm{eff}}$ son los primeros candidatos a desaparecer. Esto genera una secuencia de árboles:

$$T_0\supset T_1\supset T_2\supset\cdots\supset T_k$$

desde el árbol completo hasta árboles progresivamente más pequeños.

## Scikit-learn puede calcular los `alpha` candidatos

No es necesario probar solamente valores arbitrarios como `0.001`, `0.01` y `0.1`: el propio árbol determina los valores relevantes. El siguiente fragmento supone que ya existen `X_train` e `y_train`.

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

# Descomenta al disponer de datos de entrenamiento:
# path = clf.cost_complexity_pruning_path(X_train, y_train)
# ccp_alphas = path.ccp_alphas
# impurities = path.impurities
# print(ccp_alphas)

Una salida ilustrativa podría ser `[0.0000, 0.0021, 0.0037, 0.0062, 0.0115, 0.0278, 0.0841]`. Cada valor representa un punto relevante del proceso de poda. Se pueden entrenar y comparar modelos con esos candidatos:

In [ ]:
# Requiere ccp_alphas, X_train e y_train del ejemplo anterior.
# modelos = []
# for alpha in ccp_alphas:
#     clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
#     clf.fit(X_train, y_train)
#     modelos.append(clf)

## Relación con sobreajuste y subajuste

| `ccp_alpha` | Árbol | Sesgo | Varianza | Riesgo |
|---:|---|---|---|---|
| 0 | Muy complejo | Bajo | Alta | Sobreajuste |
| Pequeño | Moderadamente complejo | Bajo o moderado | Moderada | Puede generalizar mejor |
| Grande | Muy simple | Alto | Baja | Subajuste |

El mejor `ccp_alpha` no es necesariamente el que maximiza el *accuracy* de entrenamiento. Conviene seleccionar el valor según el desempeño en validación.

## Una distinción importante

`max_depth`, `min_samples_split`, `min_samples_leaf` y `max_leaf_nodes` restringen principalmente el crecimiento del árbol. En cambio, conceptualmente `ccp_alpha` aplica **poda posterior**:

$$\boxed{\text{construir árbol}\longrightarrow\text{evaluar subárboles}\longrightarrow\text{podar}}$$

Esta distinción ayuda a explicar en clase la diferencia entre **poda previa** y **poda posterior**.

Para un ejemplo docente, se pueden graficar simultáneamente `ccp_alpha` frente al número de nodos, profundidad, *accuracy* de entrenamiento y *accuracy* de prueba. Esas curvas permiten observar cómo cambia la complejidad y el desempeño al podar.